In [1]:
from quantrisk import ARMAGARCH, Backtest, Plotter, Risk, Statistics
import numpy as np
import pandas as pd

For this demo we will be analyzing log returns of Apple (AAPL) from 01/02/2015-05/29/2026. We will use the last 1000 trading days in our demo.

In [2]:
path = "DATA/AAPL.csv"
df = pd.read_csv(path, header=[0, 1], index_col=0, parse_dates=True)
prices = df["Close"].dropna()
returns = np.log(prices / prices.shift(1)).dropna()
df["Returns"] = returns
data = df["Returns"].iloc[-1000:]

# 1. ARMA(1,1)-GARCH(1,1)

We will first fit an ARMA(1,1)-GARCH(1,1) model with Gaussian shocks to our returns data. Later (see Sec. 2) we will see how we can model the data using shocks from other distributions. We use the following notation:
$$r_t = \mathbb{E}[r_{t}|\mathcal{F}_{t-1}] + \varepsilon_t = \xi + \phi r_{t-1} + \theta \varepsilon_{t-1} + \varepsilon_{t}$$
$$\sigma_{t}^2 = C + B \sigma_{t-1}^2 + A\varepsilon_{t-1}^2$$
$$\varepsilon_t = \sigma_t z_t$$
$$z_t \sim N(0,1)$$
Here $\mathcal{F}_t$ denotes the filtration at time $t$. We refer to $\{\varepsilon_t\}$ as the **innovations** and $\{z_t\}$ as the **shocks**. 

First create an `ARMAGARCH` object and fit the model by using the `.fit()` method.

In [3]:
AAPL_AG = ARMAGARCH()
AAPL_AG.fit(data)

The fitted ARMA and GARCH parameters are stored in a dictionary as object attributes called `ARMA_params` and `GARCH_params`, respectively.

In [4]:
print("Fitted ARMA parameters:")
for k,v in AAPL_AG.ARMA_params.items():
    print(f"{k} = {v}")

Fitted ARMA parameters:
phi = -0.12531441181056213
theta = 0.1735936554506209
xi = 0.0008376853332238213


In [5]:
print("Fitted GARCH parameters:")
for k,v in AAPL_AG.GARCH_params.items():
    print(f"{k} = {v}")

Fitted GARCH parameters:
A = 0.07036095548848638
B = 0.8927952043244026
C = 1.0765482484116214e-05


Once the model has been fit we can retrieve the innovations $\{\varepsilon_t\}$ and the forecasted variances by calling `.get_innovations()`.

In [6]:
eps_t, sigma2_t = AAPL_AG.get_innovations(data)

We can then use the innovations and the forecasted variances to obtain the shocks $\{z_t\}$ using the `.get_shocks()` method, as well as a one day forecast using the `.one_day_forecast()` method.

In [7]:
z_t = AAPL_AG.get_shocks(eps_t, sigma2_t)
forecasted_mean, forecasted_var = AAPL_AG.one_day_forecast(data)

# 2. Risk

As we have seen, `ARMAGARCH` can be used as a standalone class. However, it is most powerful when used within the `Risk` class, as we now describe. One of the advantages with the. `Risk` class is the ability to construct an ARMA(1,1)-GARCH(1,1) model with non-Gaussian errors. 

To begin, create a `Risk` object. One can pass in their desired distribution for the shocks. The currently supported distributions are: Gaussian, (location-scale) Student-t, and Normal Inverse Gaussian (NIG). One can also pass in a previously trained ARMA-GARCH model, but this. is not necessary (see README for details). 

For our demo we will use the NIG distribution.

In [8]:
AAPL_risk = Risk()